## 데이터 수집 자동화
### 셀레니움

- 웹 브라우저를 자동으로 제어하는 파이썬 라이브러리(패키지)

- 기능

- 브라우저 실행
- 웹 페이지 로드(접속)
- 입력창에 글자 입력(아이디, 패스워드 등)
- 버튼 클릭
- 페이지 이동
- 화면에 표시된 텍스트 추출
- 특정 이미지 저장 / 화면 스크린 캡처
- 반복 자동화
- 정적인 HTML 은 Beautifulsoup4, request 패키지로 수집가능

- 동적인 웹화면은 Selenium이 강력함. 정적인 웹페이지도 가능

#### 웹브라우저 실행

In [1]:
# 셀레니움 패키지 로드
from selenium import webdriver
import pandas as pd

In [2]:
# 웹브라우저 프로세스를 파이썬이 driver 변수로 관리
driver = webdriver.Chrome()

driver.get(f'http://www.naver.com')

In [3]:
# 웹브라우저 종료
driver.quit()

#### 웹 페이지 스크래핑 실습

In [8]:
driver = webdriver.Chrome()

driver.get(f'http://127.0.0.1:5500/chapt04/products.html')

##### 요소 찾기
- 주요 요소 찾기 방식
- By.ID : ID로 찾기
- By.CLASS_NAME : class명으로 찾기
- By.TAG_NAME : 태그명으로 찾기
- By.CSS_SELECTOR : CSS 선택자 방법으로 찾기
- By.XPATH : XPath로 찾기

In [6]:
# CSS 선택자로 요소찾는 함수용 패키지 내 클래스 로드
from selenium.webdriver.common.by import By

In [9]:
# 페이지 내 하나의 데이터만 추출
title = driver.find_element(By.ID, 'title')

title.text

'상품목록'

In [10]:
# 페이지 내 여러 요소를 동시에 추출
products = driver.find_elements(By.CLASS_NAME, 'product')

for product in products:
    print(product.text)

키보드 30000
마우스 15000
모니터 250000


#### CSS 선택자로 요소 추출

In [11]:
products = driver.find_elements(By.CSS_SELECTOR, '.product')

for product in products:
    print(product)
    name = product.find_element(By.CSS_SELECTOR, '.name').text
    price = product.find_element(By.CSS_SELECTOR, '.price').text
    print(name, price)

<selenium.webdriver.remote.webelement.WebElement (session="66bf31a7aaf5cdd22ac55eb07242a0a2", element="f.4AD86A4DEF288F30C76479ECB986FBDD.d.3FDD266A7FD0AED51148E9532455B81E.e.16")>
키보드 30000
<selenium.webdriver.remote.webelement.WebElement (session="66bf31a7aaf5cdd22ac55eb07242a0a2", element="f.4AD86A4DEF288F30C76479ECB986FBDD.d.3FDD266A7FD0AED51148E9532455B81E.e.17")>
마우스 15000
<selenium.webdriver.remote.webelement.WebElement (session="66bf31a7aaf5cdd22ac55eb07242a0a2", element="f.4AD86A4DEF288F30C76479ECB986FBDD.d.3FDD266A7FD0AED51148E9532455B81E.e.18")>
모니터 250000


In [12]:
keyword_input = driver.find_element(By.ID, 'keyword')
keyword_input.send_keys('USB');

In [13]:
keyword_input.clear()

In [14]:
keyword_input.send_keys('테슬라')

In [15]:
# 입력태그에서 텍스트 가져오기
keyword_str = driver.find_element(By.ID, 'keyword')

keyword_str.get_attribute('value')

'테슬라'

##### 버튼클릭

In [16]:
button = driver.find_element(By.ID, 'search-btn')
button.click()

#####명시적 대기

- 링크 클릭으로 다른 페이지 이동, 검색 후 결과 대기까지 시간이 소요
- time.sleep(2) 2초동안 대기 등의 파이썬 기본 함수가 제약사항이 많음
- 특정 조건이 만족할때까지 대기하는 로직 사용

In [17]:
# 명시적 대기를 위한 패키지 로드
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [19]:
# 여러 작업 수행 후 

wait = WebDriverWait(driver, 10)   # 10초까지 대기

title = wait.until(
    EC.presence_of_element_located((By.ID, 'title'))
)

title.text

'상품목록'

##### 데이터 수집하기

In [20]:
products = driver.find_elements(By.CLASS_NAME, 'product')

result = []

for product in products:
    name = product.find_element(By.CSS_SELECTOR, '.name').text
    price = product.find_element(By.CLASS_NAME, 'price').text

    result.append({ 'name': name, 'price': price })

In [21]:
result

[{'name': '키보드', 'price': '30000'},
 {'name': '마우스', 'price': '15000'},
 {'name': '모니터', 'price': '250000'}]

In [22]:
df = pd.DataFrame(result)
df

,name,price
0,키보드,30000
1,마우스,15000
2,모니터,250000


In [23]:
df.to_csv('../data/scrap_products.csv', encoding='utf-8')

##### 정리

- 웹 스크래핑시 필요요소
- 웹드라이버로 웹사이트 오픈
- 필요 요소 찾기
- 입력태그 찾기
- 버튼 요소 찾기
- 입력하고 버튼 클릭해서 동작
- 데이터 가져오기
-화면이 갱신되기까기 명시적 대기

##### 실습

###### 멜론 사이트 TOP100 스크래핑

In [24]:
# 패키지로드
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [25]:
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 10)
driver.get('https://www.melon.com/chart/index.htm')

company_name = wait.until(
    EC.presence_of_element_located((By.ID, 'footer'))
)

In [27]:
driver = webdriver.Chrome()

##### 차트의 곡 행 가져오기

In [28]:
# CSS 선택자 입력시 태그 + .css 동일한 기능
# songs = driver.find_elements(By.CSS_SELECTOR, '.lst50, .lst100')
songs = driver.find_elements(By.CSS_SELECTOR, 'tr.lst50, tr.lst100')

# len(songs)
for song in songs:
    # 노래 순위
    rank = song.find_element(By.CSS_SELECTOR, '.rank').text
    # 노래제목
    title = song.find_element(By.CSS_SELECTOR, '.rank01 > span > a').text
    # 가수명
    singer = song.find_element(By.CSS_SELECTOR, '.rank02 a').text    
    # 앨범명
    albumn = song.find_element(By.CSS_SELECTOR, '.rank03 a').text

    # 좋아요 개수
    like = song.find_element(By.CLASS_NAME, 'cnt').text

    # 단계상승하락
    rank_ud = song.find_element(By.CSS_SELECTOR, '.rank_wrap').get_attribute('title')

    # 이미지 썸네일 주소 / 생략
    img_url = song.find_element(By.CSS_SELECTOR, '.image_typeAll img').get_attribute('src')

    print(rank, rank_ud, title, singer, albumn, like)

In [29]:
songs = driver.find_elements(By.CSS_SELECTOR, 'tr.lst50, tr.lst100')

datas = []

# len(songs)
for song in songs:
    # 노래 순위
    rank = song.find_element(By.CSS_SELECTOR, '.rank').text
    # 노래제목
    title = song.find_element(By.CSS_SELECTOR, '.rank01 > span > a').text
    # 가수명
    singer = song.find_element(By.CSS_SELECTOR, '.rank02 a').text    
    # 앨범명
    albumn = song.find_element(By.CSS_SELECTOR, '.rank03 a').text

    # 좋아요 개수
    like = song.find_element(By.CLASS_NAME, 'cnt').text
    # 단계상승하락
    rank_ud = song.find_element(By.CSS_SELECTOR, '.rank_wrap').get_attribute('title')

    datas.append({
        '순위': rank,
        '순위변동' : rank_ud,
        '노래제목': title,
        '가수': singer,
        '앨범': albumn,
        '좋아요': like
    })

In [30]:
driver.quit()

In [31]:
df_melon100 = pd.DataFrame(datas)
df_melon100

""


In [32]:
df_melon100.to_csv('../data/멜론Top100_261001.csv', encoding='utf-8')